# Multilinear epistasis

Allelic effects are vectors in trait space; loci interact through a pairwise
multilinear term, `x = S + Σ_{i<j} ε(Y_i, Y_j)` (Hansen & Wagner 2001), where
`ε` is a `p × p × p` tensor shared by all pairs of loci and `S` is the sum of
locus effects. With `epsilon = nothing` the model is purely additive.
See `?EvoMod.Multilinear`.

This notebook compares the additive model with positive and negative
directional epistasis under directional selection on trait 1. Theory predicts
that positive directional epistasis increases the additive variance as the
population moves in the direction of selection, and negative epistasis
decreases it.

In [ ]:
using Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()

using EvoMod, EvoMod.Multilinear
using Gadfly, DataFrames, LinearAlgebra, Random, Statistics
include("plots.jl")
Random.seed!(42)

In [ ]:
# Parameters
n_e   = 1000
loci  = 50
p     = 2
mu    = 0.005    # mutation probability per allele
mut_sd = 0.05    # standard deviation of mutational effects (per trait)
v_e   = 1.0      # environmental standard deviation
e     = 0.05     # strength of directional epistasis

omega = 20 * Matrix(1.0I, p, p)
delta_theta = [0.005, 0.0]          # optimum moves along trait 1
regimes     = ["Stab", "Direct"]
generations = [1000, 2000]
thin        = 20

# epistasis of each trait with itself only: ε[k, k, k] = e
function self_epistasis(e, p)
    E = zeros(p, p, p)
    for k in 1:p
        E[k, k, k] = e
    end
    E
end

models = Dict("additive" => nothing,
              "positive" => self_epistasis(e, p),
              "negative" => self_epistasis(-e, p))

In [ ]:
pops = Dict(name => run_pop(Params(loci, p; mu = mu, mutation_cov = mut_sd, v_e = v_e, epsilon = eps),
                           n_e, regimes, generations;
                           delta_theta = delta_theta, omega = omega, thin = thin)
            for (name, eps) in models)

Additive genetic variance of trait 1 through time; directional selection starts after the stabilizing phase:

In [ ]:
df = reduce(vcat, [DataFrame(model = name,
                              gen = [m["gen"] for m in pop.moments],
                              G11 = [m["G"][1, 1] for m in pop.moments],
                              x1 = [m["mean_x"][1] for m in pop.moments])
                    for (name, pop) in pops])
plot(df, x = :gen, y = :G11, color = :model, Geom.line,
     Guide.xlabel("Generation"), Guide.ylabel("Additive variance, trait 1"))

In [ ]:
plot(df, x = :gen, y = :x1, color = :model, Geom.line,
     Guide.xlabel("Generation"), Guide.ylabel("Mean genotypic value, trait 1"))

Local effects: the effect of each locus in the background of the rest of
the genome (`local_effects`). With positive epistasis, loci that moved in the
direction of selection have larger effects than their reference effects:

In [ ]:
pop = pops["positive"]
ind = pop[1]
DataFrame(reference = locus_effects(ind.y)[1, :], local_effect = local_effects(ind.par, ind.y)[1, :])

In [ ]:
plotTraits(pops["positive"])